In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser, ListOutputParser, CommaSeparatedListOutputParser, PydanticOutputParser, JsonOutputParser
from pydantic import BaseModel, Field
from typing import List
from dotenv import find_dotenv, load_dotenv
import os

env_path = find_dotenv()
if not env_path:
    raise FileNotFoundError(".env file not found.")

load_dotenv(env_path)
apiKey = os.getenv("OLLAMA_API_KEY")
os.environ['OLLAMA_API_KEY'] = apiKey
llm = ChatOllama(model="gpt-oss:120b-cloud")

In [ ]:
prompt = PromptTemplate(
    template="Write a poem about Paris"
)
response = llm.invoke(prompt.format())
print(response.content)

In [ ]:
prompt = PromptTemplate(
    template="Write a poem about Paris"
)
parser = StrOutputParser()
chain = prompt | llm | parser
response = chain.invoke({})
print(response)

In [ ]:
class CommaSeparatedListParser(ListOutputParser):
    def parse(self, text: str):
        # Split by ',' and strip whitespace
        return [item.strip() for item in text.split(',') if item.strip()]

prompt = PromptTemplate(
    template="Give comma separated list of only names of top 5 populous cities in the world"
)
#parser = CommaSeparatedListParser()
parser = CommaSeparatedListOutputParser()
chain = prompt | llm | parser
response = chain.invoke({})
print(response)

In [ ]:
# Define schema
class Topic(BaseModel):
   description: str = Field(description="A concise description of the topic")
   hashtags: str = Field(description="Keywords in hashtag format (at least 2)")
# Initialize parser with schema
parser = JsonOutputParser(pydantic_object=Topic)
# Create model and prompt
prompt = ChatPromptTemplate.from_messages([
   ("system", "You are a helpful assistant."),
   ("user", "#Format: {format_instructions}\n\n#Question: {question}")
]).partial(format_instructions=parser.get_format_instructions())
# Chain prompt → model → parser
chain = prompt | llm | parser
# Invoke chain
result = chain.invoke({"question": "Explain the severity of global warming."})
print(result)

In [ ]:
class Person(BaseModel):
    name: str = Field(description="Person's name"),
    age: int = Field(description="Person's age"),
    hobbies: List[str] = Field(description="Person's hobbies")

parser = PydanticOutputParser(pydantic_object=Person)
prompt = PromptTemplate(
    template=(
        "Extract the person's details from the text below.\n"
        "Text: {text}\n\n"
        "{format_instructions}"
    ),
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
    )
input_prompt = prompt.format(
    text="John Doe is a 29-year-old software engineer who enjoys hiking, chess, and painting."
)
response = llm.invoke(input_prompt)
# 7. Parse the output into the Pydantic model
try:
    person = parser.parse(response.content)
    print("Parsed object:", person)
    print("Name:", person.name)
    print("Age:", person.age)
    print("Hobbies:", person.hobbies)
except Exception as e:
    print("Parsing failed:", e)